# PyDI Data Integration Workflow: Companies

This notebook demonstrates how PyDI is used for end-to-end data integration. We'll work with companies datasets to showcase the data integration pipeline from schema and entity matching to data fusion.

## Table of Contents
  - [Datasets](#datasets)
- [Part 1: Schema Matching and Value Normalization](#part-1-schema-matching-and-value-normalization)
  - [Step 1: Load Target Schema and Normalization Spec](#step-1-load-target-schema-and-normalization-spec)
  - [Step 2: Load Source Datasets](#step-2-load-source-datasets)
  - [Step 3: LLM-Based Schema Matching](#step-3-llm-based-schema-matching)
  - [Step 4: Schema Matching Evaluation](#step-4-evaluate-schema-matching-against-gold-mapping)
  - [Step 5: Translate and Normalize](#step-5-translate-and-normalize)
- [Part 2: Data Profiling](#part-2-data-profiling)
- [Part 3: Entity Matching](#part-3-entity-matching)
  - [Step 1: Blocking](#step-1-blocking)
  - [Step 2: Blocking Evaluation](#step-2-evaluate-blocking-against-ground-truth)
  - [Step 3: Entity Matching with Comparators](#step-3-entity-matching-with-comparators)
  - [Step 4: Entity Matching Evaluation](#step-4-evaluate-matching-against-ground-truth)
- [Part 4: Data Fusion](#part-4-data-fusion)
  - [Step 1: Define Fusion Strategy](#step-1-define-fusion-strategy)
  - [Step 2: Run Fusion](#step-2-run-fusion)
  - [Step 3: Data Fusion Evaluation](#step-3-evaluate-data-fusion)

## Part 1: Schema Matching and Value Normalization

In [ ]:
import time
start_time = time.time()

from pathlib import Path

# Paths relative to this notebook
NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR = NOTEBOOK_DIR / "input"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd
import json
from PyDI.schemamatching import LLMBasedSchemaMatcher, SchemaMappingEvaluator, SchemaTranslator
from PyDI.normalization import load_normalization_spec
from langchain_openai import ChatOpenAI

## Step 1: Load Target Schema and Normalization Spec

In [ ]:
# Load the JSON Schema (used for both matching and normalization)
with open(INPUT_DIR / "schemamatching" / "target_schema.json") as f:
    target_schema = json.load(f)

# Load NormalizationSpec from the same schema
spec = load_normalization_spec(INPUT_DIR / "schemamatching" / "target_schema.json")
spec.set_column("founders", output_type="list<string>")
spec.set_column("founded", output_type="string")
target_columns = list(spec.columns.keys())

# Create empty target DataFrame for schema matching
df_target = pd.DataFrame(columns=target_columns)
df_target.attrs["dataset_name"] = "target_schema"

# Show column types derived from schema
pd.DataFrame([
    {"column": col, "output_type": col_spec.output_type}
    for col, col_spec in spec.columns.items()
])

## Step 2: Load Source Datasets

In [ ]:
from PyDI.io import load_csv, load_xml, load_json
import numpy as np


def normalize_iso_date_column(df, column):
    if column not in df.columns:
        return df

    original = df[column]
    text = original.astype("string").str.strip()
    missing = original.isna() | text.eq("")
    parsed = pd.to_datetime(text, errors="coerce")
    normalized = parsed.dt.strftime("%Y-%m-%d")
    df[column] = normalized.where(parsed.notna(), original)
    df.loc[missing, column] = pd.NA
    return df


dbpedia = load_csv(INPUT_DIR / "data" / "dbpedia.csv")
dbpedia.attrs["dataset_name"] = "dbpedia"
normalize_iso_date_column(dbpedia, "established")
dbpedia["keypeople_name"] = dbpedia["keypeople_name"].apply(lambda x: x if x else np.nan)
dbpedia.head(10)

In [ ]:
forbes = load_csv(INPUT_DIR / "data" / "forbes.csv")
forbes.attrs["dataset_name"] = "forbes"
forbes.head()

In [ ]:
fullcontact = load_csv(INPUT_DIR / "data" / "fullcontact.csv")
fullcontact.attrs["dataset_name"] = "fullcontact"
normalize_iso_date_column(fullcontact, "Attribute_6")
fullcontact.head()

## Step 3: LLM-Based Schema Matching

In [ ]:
from dotenv import load_dotenv
load_dotenv()

# Initialize matcher with target schema for better context
matcher = LLMBasedSchemaMatcher(
    chat_model=ChatOpenAI(model="gpt-5"),
    num_rows=40,
    target_schema=target_schema,
)

# Match dbpedia dataset
dbpedia_mapping = matcher.match(dbpedia, df_target)

dbpedia_mapping

In [ ]:
forbes_mapping = matcher.match(forbes, df_target)
# add forbes_url as id to the mapping

forbes_mapping = pd.concat([
    forbes_mapping,
    pd.DataFrame([{
        "source_dataset": "forbes",
        "source_column": "forbes_url",
        "target_dataset": "target_schema",
        "target_column": "id",
        "score": 1.0,
        "notes": "manual",
    }]),
], ignore_index=True)

forbes_mapping

In [ ]:
fullcontact_mapping = matcher.match(fullcontact, df_target)
fullcontact_mapping

## Step 4: Evaluate Schema Matching Against Gold Mapping


In [ ]:
# Load the manually curated schema-matching gold standard
with open(INPUT_DIR / "schemamatching" / "sm_mapping_gold.json") as f:
    schema_mapping_gold = json.load(f)

gold_rows = (
    schema_mapping_gold["mappings"]
    + schema_mapping_gold.get("workflow_derived_mappings", [])
)
schema_mapping_gold_df = pd.DataFrame(gold_rows)

schema_mapping_predictions = pd.concat(
    [dbpedia_mapping, forbes_mapping, fullcontact_mapping],
    ignore_index=True,
)

schema_matching_metrics = SchemaMappingEvaluator.evaluate(
    schema_mapping_predictions,
    schema_mapping_gold_df,
    complete=True,
)

schema_matching_summary = pd.DataFrame([
    {
        **schema_matching_metrics,
        "predicted_mappings": len(schema_mapping_predictions),
        "gold_mappings": len(schema_mapping_gold_df),
    }
])

per_source_schema_matching = pd.DataFrame([
    {
        "source_dataset": source_dataset,
        **SchemaMappingEvaluator.evaluate(
            schema_mapping_predictions[
                schema_mapping_predictions["source_dataset"] == source_dataset
            ],
            schema_mapping_gold_df[
                schema_mapping_gold_df["source_dataset"] == source_dataset
            ],
            complete=True,
        ),
    }
    for source_dataset in sorted(schema_mapping_gold_df["source_dataset"].unique())
])

display(schema_matching_summary)
per_source_schema_matching


## Step 5: Translate and Normalize


In [ ]:
translator = SchemaTranslator()
# Translate + normalize each dataset with its own mapping

# Adjust forbes financials from billions to absolute numbers
#forbes["sales_figure"] = forbes["sales_figure"].apply(lambda x: x * 1e9 if pd.notna(x) else x)
#forbes["asset_value"] = forbes["asset_value"].apply(lambda x: x * 1e9 if pd.notna(x) else x)

# Remove , from dbpedia financials
dbpedia["total_assets_val"] = dbpedia["total_assets_val"].astype(str).str.replace(",", "", regex=False)
dbpedia["annual_income"] = dbpedia["annual_income"].astype(str).str.replace(",", "", regex=False)
for col in ["total_assets_val", "annual_income"]:
    dbpedia[col] = pd.to_numeric(dbpedia[col], errors="coerce")
    dbpedia[col] = dbpedia[col].round().astype("Int64")

spec.set_column("country", country_format="name")

forbes_normalized = translator.translate(
    forbes, forbes_mapping,
    normalize=spec, on_failure="keep"
)

dbpedia_normalized = translator.translate(
    dbpedia, dbpedia_mapping,
    normalize=spec, on_failure="keep"
)

fullcontact_normalized = translator.translate(
    fullcontact, fullcontact_mapping,
    normalize=spec, on_failure="keep"
)

# =========================================================================
# 3. Text structural feature isolation for model matching
# =========================================================================
import re
import unicodedata

STRUCTURAL_CLEAN_MAP = {
    "texasdallas": "dallas texas",
    "californiamilpitas california": "milpitas california",
    "texashouston": "houston texas",
    "korea republic of": "south korea",
    "republic of korea": "south korea",
    "peoples republic of china": "china",
    "hong kong china": "hong kong",
    "new york city": "new york",
    "midtown manhattan": "new york",
    "winstonsalem north california": "winstonsalem",
    "fujianlongyanshanghang": "shanghang longyan fujian"
}

def clean_structural_anomalies(text):
    if not text or pd.isna(text):
        return ""
    
    s = str(text).strip().lower()
    s = re.sub(r"\(.*?\)", "", s) # Drop trailing descriptions like (city)
    s = s.split(';')[0].strip()   # Capture text prior to semi-colon lists
    
    # Dynamic accent / diacritic stripping
    s = "".join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
    
    # Strip layout punctuation and commas
    s = re.sub(r"[^\w\s]", "", s)  
    s = re.sub(r"\s+", " ", s).strip()
    
    if s.endswith(" city"):
        s = s[:-5]
    if " bang rak district" in s:
        s = s.replace(" bang rak district", "").replace("si lom ", "")
        
    if s in STRUCTURAL_CLEAN_MAP:
        return STRUCTURAL_CLEAN_MAP[s]
        
    return s

# =========================================================================
# 4. Apply the structural cleaning function to the normalized datasets
# =========================================================================
# We iterate through the normalized datasets and apply the function
# We also use .get() to handle cases where 'city' or 'country' might be missing
for df in [forbes_normalized, dbpedia_normalized, fullcontact_normalized]:
    # Ensure columns exist before trying to apply (avoid KeyError)
    if 'city' in df.columns:
        df['match_city'] = df['city'].apply(clean_structural_anomalies)
    else:
        df['match_city'] = ""
        
    if 'country' in df.columns:
        df['match_country'] = df['country'].apply(clean_structural_anomalies)
    else:
        df['match_country'] = ""

# Force-fix DBpedia financials to actual numbers
# Using the TARGET schema names ('assets' and 'revenue') that the translator created
for df in [dbpedia_normalized]:
    for col in ['assets', 'revenue']: 
        if col in df.columns:
            # Convert to numeric, then to Int64
            df[col] = pd.to_numeric(df[col], errors='coerce').round().astype('Int64')

In [ ]:
# Inspect normalized dbpedia dataset (target columns only)
dbpedia_cols = [c for c in target_columns if c in dbpedia_normalized.columns]
dbpedia_normalized[dbpedia_cols].head()

In [ ]:
# Inspect normalized forbes dataset (target columns only)
forbes_cols = [c for c in target_columns if c in forbes_normalized.columns]
forbes_normalized[forbes_cols].head()

In [ ]:
# Inspect normalized fullcontact dataset (target columns only)
fullcontact_cols = [c for c in target_columns if c in fullcontact_normalized.columns]
fullcontact_normalized[fullcontact_cols].head()

In [ ]:
# Only keep target columns
dbpedia = dbpedia_normalized[dbpedia_cols].copy()
forbes = forbes_normalized[forbes_cols].copy()
fullcontact = fullcontact_normalized[fullcontact_cols].copy()

In [ ]:
# --- MaDI-Bench normalization output: the normalized source tables, one CSV per source
# (record id + target-schema columns), the input of madi_bench.evaluation.score_normalization,
# which scores them against input/normalization/<split>.csv.
import json as _json
from pathlib import Path as _Path
_norm_dir = OUTPUT_DIR / "normalization"
_norm_dir.mkdir(parents=True, exist_ok=True)
_target = list(_json.loads((_Path(NOTEBOOK_DIR) / "input" / "schemamatching" / "target_schema.json").read_text())["properties"])
for _name, _df in {"forbes": forbes_normalized, "dbpedia": dbpedia_normalized, "fullcontact": fullcontact_normalized}.items():
    _id_col = "id" if "id" in _df.columns else next((c for c in _df.columns if c.endswith("_id")), _df.columns[0])
    _cols = [_id_col] + [c for c in _df.columns if c in _target and c != "id" and c != _id_col]
    _df[_cols].rename(columns={_id_col: "id"}).to_csv(_norm_dir / f"{_name}.csv", index=False)
    print(f"normalized table {_name}: {len(_df)} rows, {len(_cols)} columns -> {_norm_dir / (_name + '.csv')}")


## Part 2: Data Profiling

In [ ]:
from PyDI.utils import DataProfiler

# Display basic information
datasets = [dbpedia, forbes, fullcontact]
names = ["DBpedia", "Forbes", "FullContact"]

total_records = sum(len(df) for df in datasets)
print(f"Total records across all datasets: {total_records:,}")

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(datasets, names):
    profile = profiler.summary(df) # automatically prints some statistics and returns object containing stats

display(profile)

### Attribute Coverage Analysis

In [ ]:
coverage = profiler.analyze_coverage(
    datasets=datasets,
    include_samples=True,
    sample_count=3  # Show 3 sample values per attribute
)

print("Attribute coverage across datasets:")
display(coverage)

# Identify attributes suitable for entity matching
print("\nAttributes suitable for entity matching:")
matching_attrs = coverage[coverage['datasets_with_attribute'] >= 2]['attribute'].tolist()
print(f"Attributes available in 2+ datasets: {matching_attrs}")

## Part 3: Entity Matching

### Step 1: Blocking

In [ ]:
# Set up logging
import logging

import os
os.makedirs('output/logs', exist_ok=True)

logging.basicConfig(
    level=logging.INFO, # Alternatively, use logging.DEBUG for more verbosity
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler('output/logs/pydi.log'),  # Save to file
          logging.StreamHandler()                      # Display on console
      ],
    force=True
)

In [ ]:
from PyDI.entitymatching import TokenBlocker

token_blocker_f2d = TokenBlocker(
    forbes_normalized, dbpedia_normalized,
    column='name',
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)

token_blocker_f2fc = TokenBlocker(
    forbes_normalized, fullcontact_normalized,
    column='name',
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)

### Step 2: Evaluate Blocking Against Ground Truth

In [ ]:
import pandas as pd
from PyDI.io import load_csv
from PyDI.entitymatching import EntityMatchingEvaluator
# Showcase EntityMatchingEvaluator.evaluate_blocking utility

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_dbpedia_val.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking_batched on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=token_blocker_f2d,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

In [ ]:
# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_fullcontact_val.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking_batched on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=token_blocker_f2fc,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

### Step 3: Entity Matching with Comparators

In [ ]:
from PyDI.entitymatching import StringComparator
import re

# ignore case and punctuation
def normalize_text(s: str) -> str: 
    if s is None:
        return ""
    return re.sub(r"[^\w\s]|_", "", s).lower()

comparators_f2d = [
    StringComparator(
        column='name', 
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    StringComparator(
        column='name', 
        similarity_function='levenshtein',
        preprocess=normalize_text
    ),
    StringComparator(
        column='country',
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    StringComparator(
        column='industry',
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
]

comparators_f2fc = [
    StringComparator(
        column='name', 
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    StringComparator(
        column='country',
        similarity_function='jaccard',
        preprocess=normalize_text
    )
]

Next, we setup the matcher and run the matching with our chosen best blocking method:

In [ ]:
from PyDI.entitymatching import RuleBasedMatcher

# Initialize Rule-Based Matcher
matcher = RuleBasedMatcher()

correspondences_f2d = matcher.match(
    df_left=forbes_normalized,
    df_right=dbpedia_normalized, 
    candidates=token_blocker_f2d,
    comparators=comparators_f2d,
    weights=[1.0, 1.0, 1.0, 0.3],
    threshold=0.2,
    id_column='id'
)

correspondences_f2fc = matcher.match(
    df_left=forbes_normalized,
    df_right=fullcontact_normalized, 
    candidates=token_blocker_f2fc,
    comparators=comparators_f2fc,
    weights=[1.0, 0.5],
    threshold=0.1,
    id_column='id'
)

### Step 4: Evaluate Matching Against Ground Truth

In [ ]:
gt_val = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_dbpedia_val.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_f2d,
    test_pairs=gt_val,
    out_dir=debug_output_dir
)

display(eval_results)

In [ ]:
print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_f2d,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

print(f"\n Cluster Size Distribution Results:")
display(cluster_distribution)

In [ ]:
# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_f2d,
    out_path=cluster_details_path
)

Additionally, PyDI offers 6 different post-clustering methods to "clean" clusters after entity matching. For example, if we want to enforce that each record in a dataset can only have exactly one correspondence in the other dataset, we can apply a greedy one-to-one matching, maximum bipartite matching or stable marriage matching.

In [ ]:
from PyDI.entitymatching import GreedyOneToOneMatchingAlgorithm

# use Greedy One-To-One Matching to refine results to 1:1 matches
clusterer = GreedyOneToOneMatchingAlgorithm()
correspondences_f2d = clusterer.cluster(correspondences_f2d)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_f2d,
    test_pairs=gt_val,
    out_dir=debug_output_dir
)

display(eval_results)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_f2d,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

print(f"\n Cluster Size Distribution Results:")
display(cluster_distribution)

In [ ]:
gt_val = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_fullcontact_val.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_f2fc,
    test_pairs=gt_val,
    out_dir=debug_output_dir
)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_f2fc,
)

clusterer = GreedyOneToOneMatchingAlgorithm()
correspondences_f2fc = clusterer.cluster(correspondences_f2fc)


cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_f2fc,
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_f2fc,
    test_pairs=gt_val,
    out_dir=debug_output_dir
)

Now, we can directly use the trained model with PyDIs MLBasedMatcher

## Part 4: Data Fusion

In [ ]:
forbes_normalized["forbes_id"] = forbes_normalized["id"]

# Assign trust scores to datasets
forbes_normalized.attrs["trust_score"] = 1
dbpedia_normalized.attrs["trust_score"] = 3
fullcontact_normalized.attrs["trust_score"] = 2

all_correspondences = pd.concat([correspondences_f2d, correspondences_f2fc], ignore_index=True)
print(f'Total correspondences: {len(all_correspondences):,}')

## Step 1: Define Fusion Strategy

In [ ]:
from PyDI.fusion import DataFusionStrategy, longest_string, shortest_string, union, prefer_higher_trust, voting, maximum, most_recent

strategy = DataFusionStrategy('company_fusion_strategy')

strategy.add_attribute_fuser('name', voting)
strategy.add_attribute_fuser('assets', prefer_higher_trust)
strategy.add_attribute_fuser('revenue', prefer_higher_trust)
strategy.add_attribute_fuser('founders', union)
strategy.add_attribute_fuser('founded', prefer_higher_trust)
strategy.add_attribute_fuser('country', voting)
strategy.add_attribute_fuser('city', shortest_string)

## Step 2: Run Fusion

In [ ]:
from PyDI.fusion import DataFusionEngine

engine = DataFusionEngine(strategy, debug=True, debug_format='json',debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion.jsonl")

fused = engine.run(
    datasets=[forbes_normalized, dbpedia_normalized, fullcontact_normalized],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)
print(f'Fused rows: {len(fused):,}')
display(fused.head(5))

## Step 3: Evaluate Data Fusion

In [ ]:
from PyDI.fusion import tokenized_match, year_only_match, set_equality_match, numeric_tolerance_match

strategy.add_evaluation_function("name", tokenized_match)
strategy.add_evaluation_function("assets", tokenized_match)
strategy.add_evaluation_function("revenue", numeric_tolerance_match, tolerance=0.1)
strategy.add_evaluation_function("assets", numeric_tolerance_match, tolerance=0.1)
strategy.add_evaluation_function("founders", set_equality_match)
strategy.add_evaluation_function("founded", year_only_match)
strategy.add_evaluation_function("country", tokenized_match)
strategy.add_evaluation_function("city", tokenized_match)

In [ ]:
# --- MaDI-Bench fusion evaluation: the public evaluation package scores the
# fused table on the validation and the test split -- strict comparators on the v2 gold set,
# all-gold accuracy (a gold record the pipeline did not produce counts as wrong) with its coverage.
# This is the scorer the benchmark's reference numbers were produced with; the notebook's own
# comparator registrations above are not used for scoring.
import json as _json, os as _os, sys as _sys
from pathlib import Path as _Path
try:
    from madi_bench.evaluation import score_fusion as _score_fusion
except ImportError:
    _sys.path.insert(0, _os.environ.get("MADI_BENCH_ROOT") or str(_Path(NOTEBOOK_DIR).resolve().parents[2]))
    from madi_bench.evaluation import score_fusion as _score_fusion
_fused_out = fused.copy()
# the notebook's own column names for target-schema attributes (the same aliases the benchmark's
# rescoring applies, reproduction/scoring/rescore.py): scoring is by target-schema name
for _src, _dst in (('keypeople_name', 'keypeople'), ('founders', 'keypeople')):
    if _src in _fused_out.columns and _dst not in _fused_out.columns:
        _fused_out = _fused_out.rename(columns={_src: _dst})
_fusion_dir = OUTPUT_DIR / "data_fusion"
_fusion_dir.mkdir(parents=True, exist_ok=True)
_fused_out.to_csv(_fusion_dir / "fused.csv", index=False)
_scores = {}
for _split in ("validation", "test"):
    _r = _score_fusion(NOTEBOOK_DIR, _fused_out, split=_split)
    _scores[_split] = _r
    print(f"{_split}: all-gold accuracy {_r['overall_accuracy_all_gold']:.4f} "
          f"(evaluated {_r['overall_accuracy_evaluated']:.4f} on {_r['n_gold_evaluated']}/{_r['n_gold']} gold records, "
          f"attribute cell coverage {_r['attribute_cell_coverage']:.2f})")
    for _a, _v in sorted(_r["per_attribute_accuracy_all_gold"].items()):
        print(f"    {_a:28s} {_v:.4f}")
(_fusion_dir / "fusion_scores_v2.json").write_text(_json.dumps(_scores, indent=1, default=str))


In [ ]:
final_time = time.time()
elapsed_time = final_time - start_time
print(f"\nTotal execution time: {elapsed_time:.2f} seconds")